# 10-second FMI magnitude comparison

Voltage and current magnitudes are intentionally plotted separately. Three-phase magnitude means `sqrt((a²+b²+c²)/3)`. Reference deltas use only the overlap covered by the saved Simulink/MATLAB reference.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = next(
    p
    for p in [
        Path.cwd(),
        Path.cwd().parents[2],
        Path("/Volumes/SSDMatthias/development/dpsim"),
    ]
    if (p / "dpsim-fmi").exists()
)
build = repo / "build-macos-check/fmi-validation/ten_second_examples"


def read(path):
    frame = pd.read_csv(path, skipinitialspace=True)
    frame.columns = [column.strip() for column in frame.columns]
    return frame


dp = {
    "scalar": read(build / "linear_scalar/logs/fmi_rlc.csv"),
    "three_phase": read(build / "linear_three_phase/logs/fmi_rlc_3ph.csv"),
    "generator": read(build / "nonlinear_generator/fmi_sg_nonlinear.csv"),
    "two_generators": read(build / "two_generators/fmi_two_generators.csv"),
    "two_converters": read(build / "two_converters/fmi_two_converters.csv"),
    "inverter": read(build / "mixed_inverter_50us_optimized/logs/fmi_mixed.csv"),
}
refs = {
    "scalar": read(
        repo / "dpsim-fmi/examples/linear_scalar/matlab/reference_simulink.csv"
    ),
    "three_phase": read(
        repo / "dpsim-fmi/examples/linear_three_phase/matlab/reference_simulink.csv"
    ),
    "generator": read(
        repo
        / "dpsim-fmi/examples/synchronous_generator/matlab/reference_matlab_ode15s.csv"
    ),
    "two_generators": read(
        repo / "dpsim-fmi/examples/two_generators/matlab/reference_ode15s.csv"
    ),
}


def mag(frame, columns):
    return np.sqrt(np.mean(frame[columns].to_numpy(float) ** 2, axis=1))


def reduced(time, values, limit=5000):
    index = np.linspace(0, len(time) - 1, min(limit, len(time))).astype(int)
    return time[index], values[index]

In [ ]:
def plot_series(axis, frame, columns, label):
    time, values = reduced(frame.time.to_numpy(float), mag(frame, columns))
    axis.plot(time, values, label=label)


fig, axes = plt.subplots(5, 2, figsize=(14, 18), sharex="col")
plot_series(axes[0, 0], dp["scalar"], ["u_V"], "input voltage")
axes[0, 0].set_title("Scalar voltage magnitude")
plot_series(axes[0, 1], dp["scalar"], ["i"], "current")
axes[0, 1].set_title("Scalar current magnitude")
plot_series(axes[1, 0], dp["three_phase"], ["u_a", "u_b", "u_c"], "DPsim")
axes[1, 0].set_title("Three-phase voltage magnitude")
plot_series(axes[1, 1], dp["three_phase"], ["i_a", "i_b", "i_c"], "DPsim")
axes[1, 1].set_title("Three-phase current magnitude")
plot_series(
    axes[2, 0], dp["generator"], ["v_bus_a_V", "v_bus_b_V", "v_bus_c_V"], "DPsim"
)
axes[2, 0].set_title("Generator bus-voltage magnitude")
plot_series(
    axes[2, 1], dp["generator"], ["i_line_a_A", "i_line_b_A", "i_line_c_A"], "DPsim"
)
axes[2, 1].set_title("Generator line-current magnitude")
plot_series(
    axes[3, 0],
    dp["two_generators"],
    ["pcc_voltage_0", "pcc_voltage_1", "pcc_voltage_2"],
    "PCC voltage",
)
axes[3, 0].set_title("Two generators: PCC voltage magnitude")
plot_series(
    axes[3, 1],
    dp["two_generators"],
    ["first_current_0", "first_current_1", "first_current_2"],
    "generator 1",
)
plot_series(
    axes[3, 1],
    dp["two_generators"],
    ["second_current_0", "second_current_1", "second_current_2"],
    "generator 2",
)
axes[3, 1].set_title("Two generators: current magnitudes")
plot_series(
    axes[4, 0],
    dp["two_converters"],
    ["pcc_voltage_0", "pcc_voltage_1", "pcc_voltage_2"],
    "PCC voltage",
)
axes[4, 0].set_title("Two converters: PCC voltage magnitude")
plot_series(
    axes[4, 1],
    dp["two_converters"],
    ["first_current_0", "first_current_1", "first_current_2"],
    "converter 1",
)
plot_series(
    axes[4, 1],
    dp["two_converters"],
    ["second_current_0", "second_current_1", "second_current_2"],
    "converter 2",
)
axes[4, 1].set_title("Two converters: current magnitudes")
for axis in axes.flat:
    axis.set_xlabel("time [s]")
    axis.set_ylabel("magnitude")
    axis.grid(True, alpha=0.25)
    axis.legend()
fig.tight_layout()

In [ ]:
def overlap_delta(dp_frame, ref_frame, dp_columns, ref_columns):
    dp_time = dp_frame.time.to_numpy(float)
    ref_time_name = "time_s" if "time_s" in ref_frame else "time"
    ref_time = ref_frame[ref_time_name].to_numpy(float)
    end = min(dp_time[-1], ref_time[-1])
    mask = dp_time <= end + 1e-12
    time = dp_time[mask]
    dp_value = mag(dp_frame.loc[mask], dp_columns)
    ref_value = np.interp(time, ref_time, mag(ref_frame, ref_columns))
    return time, dp_value, ref_value, dp_value - ref_value, end


comparisons = {
    "three-phase voltage": overlap_delta(
        dp["three_phase"],
        refs["three_phase"],
        ["u_a", "u_b", "u_c"],
        ["u_a_V", "u_b_V", "u_c_V"],
    ),
    "three-phase current": overlap_delta(
        dp["three_phase"],
        refs["three_phase"],
        ["i_a", "i_b", "i_c"],
        ["i_a_A", "i_b_A", "i_c_A"],
    ),
    "generator voltage": overlap_delta(
        dp["generator"],
        refs["generator"],
        ["v_bus_a_V", "v_bus_b_V", "v_bus_c_V"],
        ["v_bus_a_V", "v_bus_b_V", "v_bus_c_V"],
    ),
    "generator current": overlap_delta(
        dp["generator"],
        refs["generator"],
        ["i_line_a_A", "i_line_b_A", "i_line_c_A"],
        ["i_line_a_A", "i_line_b_A", "i_line_c_A"],
    ),
    "two-generator PCC voltage": overlap_delta(
        dp["two_generators"],
        refs["two_generators"],
        ["pcc_voltage_0", "pcc_voltage_1", "pcc_voltage_2"],
        ["v_pcc_a", "v_pcc_b", "v_pcc_c"],
    ),
    "two-generator current": overlap_delta(
        dp["two_generators"],
        refs["two_generators"],
        ["first_current_0", "first_current_1", "first_current_2"],
        ["first_i_a", "first_i_b", "first_i_c"],
    ),
}

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for axis, (label, (time, dp_value, ref_value, delta, end)) in zip(
    axes.flat, comparisons.items()
):
    t, y = reduced(time, dp_value)
    axis.plot(t, y, label="DPsim")
    t, y = reduced(time, ref_value)
    axis.plot(t, y, "--", label="Simulink/MATLAB reference")
    axis.set_title(label + f" magnitude; overlap {end:g} s")
    axis.set_xlabel("time [s]")
    axis.set_ylabel("magnitude")
    axis.grid(True, alpha=0.25)
    axis.legend()
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for axis, (label, (time, dp_value, ref_value, delta, end)) in zip(
    axes.flat, comparisons.items()
):
    t, y = reduced(time, delta)
    axis.plot(t, y)
    axis.axhline(0, color="black", linewidth=0.8)
    axis.set_title("Delta: DPsim - reference: " + label + f" ({end:g} s overlap)")
    axis.set_xlabel("time [s]")
    axis.set_ylabel("magnitude delta")
    axis.grid(True, alpha=0.25)
fig.tight_layout()
print("Maximum absolute magnitude deltas over valid reference overlap:")
for label, (time, dp_value, ref_value, delta, end) in comparisons.items():
    scale = max(np.max(np.abs(ref_value)), 1e-12)
    print(
        f"{label:28s} overlap={end:6.3f} s  max_delta={np.max(np.abs(delta)):.6g}  relative={np.max(np.abs(delta)) / scale:.6g}"
    )
converter = dp["two_converters"]
c1 = mag(converter, ["first_current_0", "first_current_1", "first_current_2"])
c2 = mag(converter, ["second_current_0", "second_current_1", "second_current_2"])
plt.figure(figsize=(13, 4))
plt.plot(converter.time, c1 - c2)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Two-converter current-magnitude delta")
plt.xlabel("time [s]")
plt.ylabel("converter 1 - converter 2")
plt.grid(True, alpha=0.25)
plt.tight_layout()
print(f"two-converter max current-magnitude delta: {np.max(np.abs(c1 - c2)):.6g}")

In [ ]:
inverter = dp["inverter"]
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
plot_series(
    axes[0],
    inverter,
    ["pcc_voltage_0", "pcc_voltage_1", "pcc_voltage_2"],
    "PCC voltage",
)
axes[0].set_title("Switched inverter: 10-second PCC voltage magnitude")
plot_series(
    axes[1],
    inverter,
    ["converter_current_0", "converter_current_1", "converter_current_2"],
    "converter current",
)
axes[1].set_title("Switched inverter: 10-second current magnitude")
axes[2].plot(inverter.time, inverter.pll_frequency, label="PLL frequency")
axes[2].plot(inverter.time, inverter.fmi_coupling_error_A, label="coupling error [A]")
axes[2].set_title("Switched inverter: scheduler diagnostics")
for axis in axes:
    axis.set_xlabel("time [s]")
    axis.grid(True, alpha=0.25)
    axis.legend()
fig.tight_layout()
print(
    "The switched-inverter run has no independent 10-second Simulink reference in the repository; this cell therefore shows the full-run DPsim result only."
)